EDA per store

Import packages

In [1]:
import pandas as pd
import sys
import matplotlib.pyplot as plt
import sklearn
import altair as alt

from sklearn.pipeline import Pipeline, make_pipeline

Import csv

In [ ]:
#df = pd.read_csv('C:/Users/J.Heuvelmans/OneDrive - Brain Research Center/Documenten/EAISI/2024Supermarket/Code/data/processed/history_per_year.csv', header=0)
df = pd.read_csv('C:/Users/J.Heuvelmans/OneDrive - Brain Research Center/Documenten/EAISI/2024Supermarket/Code/data/processed/downcasted_history_per_year.csv', header=0)

Show data

In [ ]:
df.head(5)

In [4]:
#columns_to_drop = ['day', 'year', 'month']
#df = df.drop(columns_to_drop, axis=1)

In [5]:
#df.head(5)

In [6]:
#df.info()
#df.isnull().sum()

In [7]:
for old, new in zip(['integer', 'float'], ['unsigned', 'float']):

    for col in df.select_dtypes(include=old).columns:
        
        df[col] = pd.to_numeric(df[col], downcast=new)

In [ ]:
df.info()

In [9]:
df['date'] = pd.to_datetime(df['date'])

Unique items per store

In [ ]:
def info_store(i): 
    store = i
    df_store = df[df['store_nbr']==store]
    unique_items = len(df_store['item_nbr'].unique())
    return {'store': store, 'unique_items': unique_items}

data = []
for i in range (1, 55): 
    data.append(info_store(i))

df_unique = pd.DataFrame(data)

hist = alt.Chart(df_unique).mark_bar().encode(
    alt.X('store:O', title='Store Number', sort='-y'),
    alt.Y('unique_items:Q', title='Number of unique items')
).properties(
    title='Histogram of unique items per store',
    width=600,
    height=400
)

hist.display()

Select single store or multiple stores

In [ ]:
single_store = 44
df_single_store = df[df['store_nbr']==single_store]
multiple_stores = [44,45,46,47,48,49,50,51]
df_multiple_stores = df[df['store_nbr'].isin(multiple_stores)]
df_all = df

df_single_store.info()
df_multiple_stores.info()
df_all.info()

Sales per Store per Item

In [ ]:
alt.data_transformers.enable("vegafusion")

df_chart = df_single_store
#df_chart = df_multiple_stores
#df_chart = df_all

chart = alt.Chart(df_chart).mark_line(opacity=0.5).encode(
    x='date:T',
    y='unit_sales:Q',
    color='item_nbr:N'
).properties(
    title='Unit Sales Over Time for Each Item',
    width=800,
    height=400
)

chart

Select most recent year of total dataframe

In [ ]:
#df_use = df_single_store
df_use = df_multiple_stores
#df_use = df_all

max_date = df_use['date'].max()
date_last_year = max_date - pd.Timedelta(days=365)
df_recent_year = df_use[df_use['date'] >= date_last_year]
df_recent_year

In [ ]:
df_recent_year.info()

In [ ]:
chart = alt.Chart(df_recent_year).mark_line(opacity=0.5).encode(
    x='date:T',
    y='unit_sales:Q',
    color='item_nbr:N'
).properties(
    title='Unit Sales Over Time for Each Item',
    width=800,
    height=400
)

chart

Calculate total days per store and plot it

In [ ]:
df_total_sales_per_day_per_store = df_recent_year.groupby(['date', 'store_nbr'])['unit_sales'].sum().reset_index()

df_total_sales_per_day_per_store

In [ ]:
#df_chart = df_single_store
#df_chart = df_multiple_stores
#df_chart = df_all
df_chart = df_total_sales_per_day_per_store

chart = alt.Chart(df_chart).mark_line(opacity=0.3).encode(
    x='date:T',
    y='unit_sales:Q',
    color='store_nbr:N'
).properties(
    title='Total Sales per Day per Store',
    width=800,
    height=800
)

chart